In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path
import requests

DATASET = "vs30"
DESTINO = Path("raw/vs30")
LIMA_BBOX = (-77.25, -12.55, -76.60, -11.70)
URL_VS30 = "https://apps.usgs.gov/shakemap_geodata/vs30/global_vs30.grd"


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total, hecho = int(r.headers.get("Content-Length", 0)), 0
        with open(destino, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
                hecho += len(bloque)
                seg = time.perf_counter() - t0
                print(f"\r  ↓ {destino.name}: {hecho/1e6:,.1f}/{total/1e6:,.1f} MB "
                      f"| {seg:,.0f} s | {hecho/1e6/max(seg,0.01):,.2f} MB/s", end="")
    print(f"\n  ✓ {destino.name} en {time.perf_counter()-t0:,.1f} s")


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

descargar(URL_VS30, DESTINO / "global_vs30.grd")

try:
    import rasterio
    from rasterio.windows import from_bounds
    with rasterio.open(DESTINO / "global_vs30.grd") as src:
        ventana = from_bounds(*LIMA_BBOX, transform=src.transform)
        datos = src.read(1, window=ventana)
        perfil = src.profile.copy()
        perfil.update(driver="GTiff", height=datos.shape[0], width=datos.shape[1],
                      transform=src.window_transform(ventana), crs=src.crs or "EPSG:4326",
                      tiled=False)
        perfil.pop("blockxsize", None); perfil.pop("blockysize", None)
        with rasterio.open(DESTINO / "vs30_lima.tif", "w", **perfil) as dst:
            dst.write(datos, 1)
    print("  ✓ Recorte de Lima creado: vs30_lima.tif")
except ImportError:
    print("  (instala rasterio para el recorte de Lima)")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path
from huggingface_hub import snapshot_download

DATASET = "xbd"
DESTINO = Path("raw/xbd")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

snapshot_download(
    repo_id="hannan022/xview2-xbd",
    repo_type="dataset",
    local_dir=DESTINO,
    max_workers=4,
)

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")
print("Licencia: CC BY-NC-SA 4.0 (uso académico). Citar: Gupta et al., 2019")

In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
import requests

DATASET = "open_buildings"
DESTINO = Path("raw/open_buildings")
LIMA_BBOX = (-77.25, -12.55, -76.60, -11.70)


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total, hecho = int(r.headers.get("Content-Length", 0)), 0
        with open(destino, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
                hecho += len(bloque)
                seg = time.perf_counter() - t0
                print(f"\r  ↓ {destino.name}: {hecho/1e6:,.1f}/{total/1e6:,.1f} MB "
                      f"| {seg:,.0f} s | {hecho/1e6/max(seg,0.01):,.2f} MB/s", end="")
    print(f"\n  ✓ {destino.name} en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def punto_en_poligono(x, y, anillo):
    dentro, j = False, len(anillo) - 1
    for i in range(len(anillo)):
        xi, yi, xj, yj = anillo[i][0], anillo[i][1], anillo[j][0], anillo[j][1]
        if (yi > y) != (yj > y) and x < (xj - xi) * (y - yi) / (yj - yi) + xi:
            dentro = not dentro
        j = i
    return dentro


def cubre_lima(geom):
    x0, y0, x1, y1 = LIMA_BBOX
    polys = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    puntos = [(x0, y0), (x0, y1), (x1, y0), (x1, y1), ((x0 + x1) / 2, (y0 + y1) / 2)]
    for poly in polys:
        anillo = poly[0]
        xs, ys = [p[0] for p in anillo], [p[1] for p in anillo]
        if max(xs) < x0 or min(xs) > x1 or max(ys) < y0 or min(ys) > y1:
            continue
        if any(punto_en_poligono(x, y, anillo) for x, y in puntos):
            return True
        if any(x0 <= x <= x1 and y0 <= y <= y1 for x, y in anillo):
            return True
    return False


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

tiles = requests.get("https://openbuildings-public-dot-gweb-research.uw.r.appspot.com/public/tiles.geojson", timeout=120).json()
seleccion = [t for t in tiles["features"] if cubre_lima(t["geometry"])]
print(f"  Teselas que cubren Lima: {len(seleccion)}")

for t in seleccion:
    p = t["properties"]
    url = p.get("tile_url") or ("https://storage.googleapis.com/open-buildings-data/v3/"
                                f"polygons_s2_level_4_gzip/{p['tile_id']}_buildings.csv.gz")
    descargar(url, DESTINO / Path(urlparse(url).path).name)

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path
import requests

DATASET = "microzonificacion"
DESTINO = Path("raw/microzonificacion")

DISTRITOS = {
    "cercado_de_lima": 197,
    "comas": 642,
    "la_molina": 565,
    "villa_el_salvador": 414,
}


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

for distrito, doc_id in DISTRITOS.items():
    try:
        descargar(f"https://sigrid.cenepred.gob.pe/sigridv3/documento/{doc_id}/descargar",
                  DESTINO / f"microzonificacion_{distrito}.pdf")
    except requests.RequestException as e:
        print(f"  ! {distrito}: {e}")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
import csv, re, time
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin
import requests

DATASET = "cismid"
DESTINO = Path("raw/cismid")
BASE = "https://www.cismid.uni.edu.pe/ceois/redacis/red/"
LIMA_BBOX = (-77.25, -12.55, -76.60, -11.70)
HEADERS = {"User-Agent": "Mozilla/5.0"}

SISMOS = {
    "2007-08-15_pisco_Mw7.9":       "event=&codef=2007&evenf=1&gid=512",
    "2025-06-15_callao_M6.1":       "event=225ewhCWcB&codef=&evenf=&gid=3226",
    "2026-05-19_ica_M6.1":          "event=22pxfhs5Ub&codef=&evenf=&gid=3812",
    "2022-01-07_lima_NE_M5.6":      "event=sgxYMOlDXm&codef=&evenf=&gid=1415",
    "2021-11-28_callao_O_M5.2":     "event=mKRyTvgpPv&codef=&evenf=&gid=1402",
}


def coordenadas_estacion(texto):
    c = re.search(r"COORDINATES\s*:\s*(-?[\d.]+)\s*,\s*(-?[\d.]+)", texto)
    if c:
        return float(c.group(1)), float(c.group(2))
    lat = re.search(r"LATITUDE\s*:\s*(-?[\d.]+)", texto)
    lon = re.search(r"LONGITUDE\s*:\s*(-?[\d.]+)", texto)
    return (float(lat.group(1)), float(lon.group(1))) if lat and lon else None


def en_lima(texto):
    coords = coordenadas_estacion(texto)
    if coords is None:
        return True
    lat, lon = coords
    x0, y0, x1, y1 = LIMA_BBOX
    return x0 <= lon <= x1 and y0 <= lat <= y1


def descargar(url, carpeta):
    t0 = time.perf_counter()
    r = requests.get(url, timeout=120, headers=HEADERS)
    r.raise_for_status()
    nombre = re.search(r'filename="?([^";]+)', r.headers.get("Content-Disposition", ""))
    nombre = nombre.group(1) if nombre else url.rsplit("=", 1)[-1] + ".txt"
    destino = carpeta / nombre
    if not en_lima(r.content[:4000].decode("utf-8", "replace")):
        print(f"  - fuera de Lima: {nombre}")
        return
    destino.write_bytes(r.content)
    print(f"  ✓ {nombre} ({len(r.content)/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

for carpeta, params in SISMOS.items():
    print(f"\n  Sismo {carpeta}")
    html = requests.get(f"{BASE}sismo.php?{params}", timeout=60, headers=HEADERS).text
    enlaces = list(dict.fromkeys(re.findall(r"href='([^']*download\.php\?[^']+)'", html)))
    enlaces = [e for e in enlaces if "CEOIS" not in e]
    print(f"  Estaciones con registro: {len(enlaces)}")
    (DESTINO / carpeta).mkdir(parents=True, exist_ok=True)
    for href in enlaces:
        try:
            descargar(urljoin(BASE, href), DESTINO / carpeta)
        except requests.RequestException as e:
            print(f"  ! {href}: {e}")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [5]:
import csv, sys, time
from datetime import datetime
from pathlib import Path
import gdown

DATASET = "stead"
DESTINO = Path("raw/stead")
PARTES = [a for a in sys.argv[1:] if a.startswith("chunk")] or ["chunk2"]
DRIVE_IDS = {
    "chunk1": "1sgLd3eVIrTMcuWtOQRN1fGmiNQ7vc227",
    "chunk2": "1ONNNtiOwFodAAeV-ZWp_-gtWhRcYgU_E",
    "chunk3": "15bt3WS-mQJmkVY4VOb5LYk45xQBYtkoM",
    "chunk4": "12Rw-WI9T21g0hrqtF1KiETYzV4rz177k",
    "chunk5": "1rR6bbqu1yE8N_EVIb3kH3wzPPVLFerFX",
    "chunk6": "1QfONZf8TBuQlkvKjwJgVMgZub4RDUkH9",
}


def descargar(drive_id, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    gdown.download(id=drive_id, output=str(destino), quiet=False)
    print(f"  ✓ {destino.name} en {(time.perf_counter()-t0)/60:,.1f} min")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S} | partes: {PARTES}")

for parte in PARTES:
    descargar(DRIVE_IDS[parte], DESTINO / f"{parte}.zip")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

⏱  Inicio: 23:04:40 | partes: ['-f', 'C:\\Users\\andre\\AppData\\Roaming\\jupyter\\runtime\\kernel-d0906ae1-a2bd-48c1-b9b2-28292c7ad379.json']
  ↓ -f.zip: 0.00/0.00 GB | 0.0 min | 0.01 MB/s
  ✓ -f.zip en 0.0 min
  ↓ kernel-d0906ae1-a2bd-48c1-b9b2-28292c7ad379.json.zip: 0.00/0.00 GB | 0.0 min | 0.01 MB/s
  ✓ kernel-d0906ae1-a2bd-48c1-b9b2-28292c7ad379.json.zip en 0.0 min
⏱  TOTAL stead: 0.05 min | 0.0 MB | 0.00 MB/s


In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path
import requests

DATASET = "limites_inei"
DESTINO = Path("raw/limites_inei")
URL = "https://raw.githubusercontent.com/juaneladio/peru-geojson/master/peru_distrital_simple.geojson"


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

descargar(URL, DESTINO / "peru_distrital_simple.geojson")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path
import geopandas as gpd
import rasterio
import requests
from rasterio.enums import Resampling
from shapely.geometry import box

DATASET = "open_buildings_25d"
DESTINO = Path("raw/open_buildings_25d")
BUCKET = "https://storage.googleapis.com/open-buildings-temporal-data"
ANIOS = [f"{a}_06_30" for a in range(2016, 2024)]
TOKENS_S2 = ["91", "97", "bd"]
RES_M = 4.0
LIMITES = Path("raw/limites_inei/peru_distrital_simple.geojson")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def descargar_altura(url, destino):
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with rasterio.open(f"/vsicurl/{url}") as src:
        factor = int(RES_M / src.res[0])
        alto, ancho = src.height // factor, src.width // factor
        altura = src.read(2, out_shape=(alto, ancho), resampling=Resampling.nearest)
        perfil = dict(driver="GTiff", height=alto, width=ancho, count=1, dtype="float32",
                      crs=src.crs, transform=src.transform * src.transform.scale(factor, factor),
                      nodata=src.nodata, compress="deflate", predictor=3, tiled=True,
                      blockxsize=512, blockysize=512)
    destino.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(destino, "w", **perfil) as dst:
        dst.write(altura, 1)
        dst.set_band_description(1, "building_height_m")
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

distritos = gpd.read_file(LIMITES)
lima_utm = distritos[distritos.NOMBPROV.isin(["LIMA", "CALLAO"])].to_crs(32718).union_all()

teselas = []
for anio in ANIOS:
    for token in TOKENS_S2:
        r = requests.get(f"{BUCKET}/v1/manifests/{token}_EPSG_32718_{anio}.json", timeout=60)
        if r.status_code != 200:
            continue
        m = r.json()
        prefijo = m["uriPrefix"].replace("gs://open-buildings-temporal-data", BUCKET)
        for ts in m["tilesets"]:
            for s in ts["sources"]:
                a, d = s["affineTransform"], s["dimensions"]
                caja = box(a["translateX"], a["translateY"] + a["scaleY"] * d["height"],
                           a["translateX"] + a["scaleX"] * d["width"], a["translateY"])
                if caja.intersects(lima_utm):
                    teselas.append(prefijo + s["uris"][0])
print(f"  Teselas que tocan Lima y Callao: {len(teselas)}")

for url in teselas:
    carpeta, archivo = url.split("/")[-2:]
    descargar_altura(url, DESTINO / f"{carpeta}_{archivo}")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")
print("Licencia: CC BY 4.0. Citar: Sirko et al., 2023 (Open Buildings 2.5D Temporal)")

In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path

DATASET = "norma_e030"
DESTINO = Path("raw/norma_e030")

ZONAS = [(4, 0.45), (3, 0.35), (2, 0.25), (1, 0.10)]

FACTOR_S = {
    4: {"S0": 0.80, "S1": 1.00, "S2": 1.05, "S3": 1.10},
    3: {"S0": 0.80, "S1": 1.00, "S2": 1.15, "S3": 1.20},
    2: {"S0": 0.80, "S1": 1.00, "S2": 1.20, "S3": 1.40},
    1: {"S0": 0.80, "S1": 1.00, "S2": 1.60, "S3": 2.00},
}

PERFILES = [
    ("S0", "Roca dura",                  1500.0,   None,     0.3, 3.0),
    ("S1", "Roca o suelos muy rígidos",   500.0,  1500.0,    0.4, 2.5),
    ("S2", "Suelos intermedios",          180.0,   500.0,    0.6, 2.0),
    ("S3", "Suelos blandos",              None,    180.0,    1.0, 1.6),
]


def escribir(nombre, cabecera, filas):
    with open(DESTINO / nombre, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(cabecera)
        w.writerows(filas)
    print(f"  ✓ {nombre} ({len(filas)} filas)")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([f"{DATASET} (transcrito)", datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 4), round(mb / max(seg, 0.01), 4)])


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

escribir("e030_zonas.csv", ["zona", "Z_g"], ZONAS)
escribir("e030_factor_suelo.csv", ["zona", "perfil", "S"],
         [(z, p, s) for z, fila in FACTOR_S.items() for p, s in fila.items()])
escribir("e030_perfiles.csv", ["perfil", "descripcion", "vs30_min", "vs30_max", "TP_s", "TL_s"], PERFILES)

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg:,.2f} s | {mb*1e3:,.1f} KB")

In [ ]:
import csv, time, zipfile
from datetime import datetime
from pathlib import Path
import requests

DATASET = "unosat_ecuador"
DESTINO = Path("raw/portoviejo_2016/unosat")
URL = "https://unosat-maps.web.cern.ch/unosat-maps/EC/EQ20160417ECU/EQ20160417ECU_shp.zip"
HEADERS = {"User-Agent": "Mozilla/5.0"}


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    parte = destino.with_name(destino.name + ".part")
    with requests.get(url, stream=True, timeout=120, headers=HEADERS) as r:
        r.raise_for_status()
        with open(parte, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    parte.replace(destino)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def extraer(zip_path, carpeta):
    raiz = carpeta.resolve()
    with zipfile.ZipFile(zip_path) as z:
        for nombre in z.namelist():
            if not (raiz / nombre).resolve().is_relative_to(raiz):
                raise ValueError(f"Ruta insegura en el zip: {nombre}")
        z.extractall(carpeta)


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

archivo = DESTINO / Path(URL).name
descargar(URL, archivo)
extraer(archivo, DESTINO / "shp")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")
print("Citar: UNITAR-UNOSAT, Ecuador 2016 (HDX).")

In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path
import requests

DATASET = "shakemap_eventos"
DESTINO = Path("raw")
EVENTOS = {"us20005j32": "raw/portoviejo_2016/shakemap"}
HEADERS = {"User-Agent": "Mozilla/5.0"}


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    parte = destino.with_name(destino.name + ".part")
    with requests.get(url, stream=True, timeout=120, headers=HEADERS) as r:
        r.raise_for_status()
        with open(parte, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    parte.replace(destino)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


mb_ini = sum(tamano_mb(c) for c in EVENTOS.values() if Path(c).exists())
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

for evento_id, carpeta in EVENTOS.items():
    evento = requests.get("https://earthquake.usgs.gov/fdsnws/event/1/query",
                          params={"eventid": evento_id, "format": "geojson"}, timeout=60, headers=HEADERS).json()
    shakemap = evento["properties"]["products"]["shakemap"][0]
    print(f"  {evento['properties']['title']} | ShakeMap versión {shakemap['properties'].get('version')}")
    descargar(shakemap["contents"]["download/grid.xml"]["url"], Path(carpeta) / "grid.xml")

seg = time.perf_counter() - inicio
mb = sum(tamano_mb(c) for c in EVENTOS.values()) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
import geopandas as gpd
import requests
from shapely.geometry import shape

DATASET = "open_buildings_portoviejo"
DESTINO = Path("raw/portoviejo_2016/open_buildings")
ANALISIS = Path("raw/portoviejo_2016/unosat/shp/PL_20160419_Portoviejo_Analysis.shp")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def descargar(url, destino):
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    parte = destino.with_name(destino.name + ".part")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(parte, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    parte.replace(destino)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

zona = gpd.read_file(ANALISIS).to_crs(4326).union_all()
tiles = requests.get("https://openbuildings-public-dot-gweb-research.uw.r.appspot.com/public/tiles.geojson", timeout=120).json()
seleccion = [t for t in tiles["features"] if shape(t["geometry"]).intersects(zona)]
print(f"  Teselas que cubren Portoviejo: {len(seleccion)}")
for t in seleccion:
    p = t["properties"]
    url = p.get("tile_url") or ("https://storage.googleapis.com/open-buildings-data/v3/"
                                f"polygons_s2_level_4_gzip/{p['tile_id']}_buildings.csv.gz")
    descargar(url, DESTINO / Path(urlparse(url).path).name)

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path
import geopandas as gpd
import rasterio
import requests
from rasterio.enums import Resampling
from shapely.geometry import box

DATASET = "open_buildings_25d_portoviejo"
DESTINO = Path("raw/portoviejo_2016/open_buildings_25d")
ANALISIS = Path("raw/portoviejo_2016/unosat/shp/PL_20160419_Portoviejo_Analysis.shp")
BUCKET = "https://storage.googleapis.com/open-buildings-temporal-data"
ANIOS = [f"{a}_06_30" for a in range(2016, 2024)]
EPSG = 32717
RES_M = 4.0


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def descargar_altura(url, destino):
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with rasterio.open(f"/vsicurl/{url}") as src:
        factor = int(RES_M / src.res[0])
        alto, ancho = src.height // factor, src.width // factor
        altura = src.read(2, out_shape=(alto, ancho), resampling=Resampling.nearest)
        perfil = dict(driver="GTiff", height=alto, width=ancho, count=1, dtype="float32",
                      crs=src.crs, transform=src.transform * src.transform.scale(factor, factor),
                      nodata=src.nodata, compress="deflate", predictor=3, tiled=True,
                      blockxsize=512, blockysize=512)
    destino.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(destino, "w", **perfil) as dst:
        dst.write(altura, 1)
        dst.set_band_description(1, "building_height_m")
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

zona = gpd.read_file(ANALISIS).to_crs(EPSG).union_all()

tokens = []
for n in range(256):
    token = f"{n:02x}"
    if requests.head(f"{BUCKET}/v1/manifests/{token}_EPSG_{EPSG}_{ANIOS[0]}.json", timeout=30).status_code == 200:
        tokens.append(token)
print(f"  Celdas S2 con UTM 17S: {tokens}")

teselas = []
for anio in ANIOS:
    for token in tokens:
        r = requests.get(f"{BUCKET}/v1/manifests/{token}_EPSG_{EPSG}_{anio}.json", timeout=60)
        if r.status_code != 200:
            continue
        m = r.json()
        prefijo = m["uriPrefix"].replace("gs://open-buildings-temporal-data", BUCKET)
        for ts in m["tilesets"]:
            for s in ts["sources"]:
                a, d = s["affineTransform"], s["dimensions"]
                caja = box(a["translateX"], a["translateY"] + a["scaleY"] * d["height"],
                           a["translateX"] + a["scaleX"] * d["width"], a["translateY"])
                if caja.intersects(zona):
                    teselas.append(prefijo + s["uris"][0])
print(f"  Teselas que tocan Portoviejo: {len(teselas)}")

for url in teselas:
    carpeta, archivo = url.split("/")[-2:]
    descargar_altura(url, DESTINO / f"{carpeta}_{archivo}")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")
print("Licencia: CC BY 4.0. Citar: Sirko et al., 2023 (Open Buildings 2.5D Temporal)")

In [ ]:
import csv, time, zipfile
from datetime import datetime
from pathlib import Path
import requests

DATASET = "ondas_portoviejo"
DESTINO = Path("raw/portoviejo_2016/ondas")
HEADERS = {"User-Agent": "grupo06-proyecto-academico"}


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    parte = destino.with_name(destino.name + ".part")
    with requests.get(url, stream=True, timeout=120, headers=HEADERS) as r:
        r.raise_for_status()
        with open(parte, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    parte.replace(destino)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def extraer(zip_path, carpeta):
    raiz = carpeta.resolve()
    with zipfile.ZipFile(zip_path) as z:
        for nombre in z.namelist():
            if not (raiz / nombre).resolve().is_relative_to(raiz):
                raise ValueError(f"Ruta insegura en el zip: {nombre}")
        z.extractall(carpeta)


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

BASE = "https://purr.purdue.edu/publications/2727/serve/1"
descargar(f"{BASE}/45395?el=3&download=1", DESTINO / "portoviejo_ground-m_data.zip")
descargar(f"{BASE}/45392?el=3&download=1", DESTINO / "readme_parameter_columns.txt")
extraer(DESTINO / "portoviejo_ground-m_data.zip", DESTINO / "datos")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")
print("Licencia: CC0 1.0. Citar: DOI 10.4231/R7ZC8111")

In [ ]:
import csv, time, zipfile
from datetime import datetime
from pathlib import Path
import requests

DATASET = "ondas_manta"
DESTINO = Path("raw/manta_2016/ondas")
HEADERS = {"User-Agent": "grupo06-proyecto-academico"}


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    parte = destino.with_name(destino.name + ".part")
    with requests.get(url, stream=True, timeout=120, headers=HEADERS) as r:
        r.raise_for_status()
        with open(parte, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    parte.replace(destino)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def extraer(zip_path, carpeta):
    raiz = carpeta.resolve()
    with zipfile.ZipFile(zip_path) as z:
        for nombre in z.namelist():
            if not (raiz / nombre).resolve().is_relative_to(raiz):
                raise ValueError(f"Ruta insegura en el zip: {nombre}")
        z.extractall(carpeta)


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

descargar("https://purr.purdue.edu/publications/2727/serve/1/45394?el=3&download=1", DESTINO / "manta_ground-m_data.zip")
extraer(DESTINO / "manta_ground-m_data.zip", DESTINO / "datos")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")
print("Licencia: CC0 1.0. Citar: DOI 10.4231/R7ZC8111")

In [ ]:
import csv, json, time, zipfile
from datetime import datetime
from pathlib import Path
import requests

DATASET = "portoviejo_campo"
DESTINO = Path("raw/portoviejo_2016/campo")
URL = "https://purr.purdue.edu/publications/2727/serve/1/45391?el=3&download=1"
HEADERS = {"User-Agent": "grupo06-proyecto-academico"}


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    parte = destino.with_name(destino.name + ".part")
    with requests.get(url, stream=True, timeout=120, headers=HEADERS) as r:
        r.raise_for_status()
        with open(parte, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    parte.replace(destino)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def extraer(zip_path, carpeta):
    raiz = carpeta.resolve()
    with zipfile.ZipFile(zip_path) as z:
        for nombre in z.namelist():
            if not (raiz / nombre).resolve().is_relative_to(raiz):
                raise ValueError(f"Ruta insegura en el zip: {nombre}")
        z.extractall(carpeta)


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

descargar(URL, DESTINO / "parameters_priority_index_metadata.csv")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
import csv, json, time, zipfile
from datetime import datetime
from pathlib import Path
import requests

DATASET = "osm_niveles_ciudades"
DESTINO = Path("raw/osm_niveles")
ZONAS = {"portoviejo": (-1.118, -80.51, -0.988, -80.411)}
HEADERS = {"User-Agent": "grupo06-proyecto-academico"}


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    parte = destino.with_name(destino.name + ".part")
    with requests.get(url, stream=True, timeout=120, headers=HEADERS) as r:
        r.raise_for_status()
        with open(parte, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    parte.replace(destino)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def extraer(zip_path, carpeta):
    raiz = carpeta.resolve()
    with zipfile.ZipFile(zip_path) as z:
        for nombre in z.namelist():
            if not (raiz / nombre).resolve().is_relative_to(raiz):
                raise ValueError(f"Ruta insegura en el zip: {nombre}")
        z.extractall(carpeta)


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

for nombre, (s, o, n, e) in ZONAS.items():
    destino = DESTINO / f"{nombre}_building_levels.json"
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        continue
    consulta = f'[out:json][timeout:180];(way["building"]["building:levels"]({s},{o},{n},{e});relation["building"]["building:levels"]({s},{o},{n},{e}););out tags center;'
    r = requests.post("https://overpass-api.de/api/interpreter", data={"data": consulta}, timeout=240, headers=HEADERS)
    r.raise_for_status()
    destino.write_bytes(r.content)
    print(f"  ✓ {destino.name}: {len(r.json()['elements'])} edificios")
    time.sleep(10)

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
import csv, time
from datetime import datetime
from pathlib import Path
import requests

DATASET = "tainan_2016"
DESTINO = Path("raw/tainan_2016")
UA = {"User-Agent": "grupo06-proyecto-academico"}
B = "https://purr.purdue.edu/publications/2748/serve"
ARCHIVOS = {"edificios/parameters_taiwan_eq_20170828.csv": f"{B}?vid=1805&file=parameters_taiwan_eq_20170828.csv&download=1",
            "edificios/1267_file_metadata.csv": f"{B}/1/45499?el=3&download=1",
            "edificios/2016_Taiwan_Meinong_20170814.csv": f"{B}/1/45500?el=3&download=1",
            "ondas/list_of_ground_motion.xlsx": f"{B}/1/45501?el=3&download=1",
            "ondas/list_of_ground_motion_Tainan_City.xlsx": f"{B}/1/45502?el=3&download=1",
            "ondas/Taiwan_CWB_ground_motion_data.zip": f"{B}/1/45503?el=3&download=1"}


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"), round(seg, 1), round(seg / 60, 2), round(mb, 3), round(mb / max(seg, 0.01), 3)])


t0 = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")
total = 0
for nombre, url in ARCHIVOS.items():
    p = DESTINO / nombre
    p.parent.mkdir(parents=True, exist_ok=True)
    if p.exists() and p.stat().st_size > 0:
        print(f"  ya existe {nombre}")
    else:
        with requests.get(url, headers=UA, stream=True, timeout=180) as r:
            r.raise_for_status()
            with open(p, "wb") as f:
                for b in r.iter_content(1 << 20):
                    f.write(b)
    total += p.stat().st_size
    print(f"  ✓ {nombre}: {p.stat().st_size / 1e6:,.2f} MB")
seg = time.perf_counter() - t0
registrar_tiempo(seg, total / 1e6)
print(f"⏱  TOTAL {DATASET}: {seg / 60:,.2f} min | {total / 1e6:,.1f} MB")

In [ ]:
import csv, hashlib, time, zipfile
from datetime import datetime
from pathlib import Path
import requests

DATASET = "turquia_2023"
DESTINO = Path("raw/turquia_2023")
HEADERS = {"User-Agent": "grupo06-proyecto-academico"}
URL_ZIP = "https://zenodo.org/api/records/18437501/files/2023Turkey_earthquake_data.zip/content"
MD5_ZIP = "f3a3a9982cf63bd954616a898c7416b9"
CARPETAS_ZIP = ("/Building_damage_data/", "/GBA_building_footprint/")
EVENTO_USGS = "us6000jllz"


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    parte = destino.with_name(destino.name + ".part")
    with requests.get(url, stream=True, timeout=120, headers=HEADERS) as r:
        r.raise_for_status()
        with open(parte, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    parte.replace(destino)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def md5(archivo):
    h = hashlib.md5()
    with open(archivo, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


def extraer(zip_path, carpeta, partes):
    raiz = carpeta.resolve()
    with zipfile.ZipFile(zip_path) as z:
        nombres = [n for n in z.namelist() if any(p in n for p in partes) and not n.endswith("/")]
        for nombre in nombres:
            if not (raiz / nombre).resolve().is_relative_to(raiz):
                raise ValueError(f"Ruta insegura en el zip: {nombre}")
        z.extractall(carpeta, members=[n for n in nombres if not (carpeta / n).exists()])
    return len(nombres)


DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

zip_tr = DESTINO / "2023Turkey_earthquake_data.zip"
descargar(URL_ZIP, zip_tr)
if md5(zip_tr) != MD5_ZIP:
    raise ValueError(f"{zip_tr.name} no coincide con el MD5 de Zenodo: bórrelo y vuelva a descargarlo")
print(f"  ✓ {extraer(zip_tr, DESTINO / 'dano', CARPETAS_ZIP)} archivos de daño y huellas en {(DESTINO / 'dano').as_posix()}")

evento = requests.get("https://earthquake.usgs.gov/fdsnws/event/1/query",
                      params={"eventid": EVENTO_USGS, "format": "geojson"}, timeout=60, headers=HEADERS).json()
shakemap = evento["properties"]["products"]["shakemap"][0]
print(f"  {evento['properties']['title']} | ShakeMap versión {shakemap['properties'].get('version')}")
for nombre in ("grid.xml", "uncertainty.xml"):
    descargar(shakemap["contents"][f"download/{nombre}"]["url"], DESTINO / "shakemap" / nombre)

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")
print("Licencias: CC BY 4.0 (citar DOI 10.5281/zenodo.18437501) y dominio público (USGS)")